<a href="https://colab.research.google.com/github/saphixe/ALLM-A2/blob/main/ALLM_A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

!pip -q install uv

# Clone BALROG only if missing
if not os.path.exists("/content/BALROG"):
    !git clone https://github.com/balrog-ai/BALROG.git

# Install Python 3.10 and create the environment if missing
if not os.path.exists("/content/balrog-venv/bin/python"):
    !uv python install 3.10
    !uv venv /content/balrog-venv --python 3.10

    # Install BALROG
    %cd /content/BALROG
    !uv pip install --python /content/balrog-venv/bin/python -e .
    !uv pip install --python /content/balrog-venv/bin/python "setuptools==80.9.0"
    !/content/balrog-venv/bin/balrog-post-install

%cd /content/BALROG

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 59.0 MB/s eta 0:00:00
Cloning into 'BALROG'...
remote: Enumerating objects: 342, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 342 (delta 166), reused 105 (delta 105), pack-reused 124 (from 1)
Receiving objects: 100% (342/342), 1.57 MiB | 4.75 MiB/s, done.
Resolving deltas: 100% (174/174), done.
Installed Python 3.10.22 in 957ms
 + cpython-3.10.22-linux-x86_64-gnu (python3.10)
Using CPython 3.10.22
Creating virtual environment at: balrog-venv
Activate with: source balrog-venv/bin/activate
/content/BALROG
Using Python 3.10.22 environment at: /content/balrog-venv
Resolved 123 packages in 8.58s
Prepared 119 packages in 33.52s
Installed 123 packages in 349ms
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.15.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + baba==0.0.1 (from git+https://github.com/nacloos/baba-is-ai.git@40ae1b29dbc67f2ecea

# Testing Ollama

In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 93 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (430 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Ollama server process started")

Ollama server process started


In [8]:
!ollama pull qwen2.5:3b

# Running qwen2.5:3b model

In [9]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = ollama_client.chat.completions.create(
    model="qwen2.5:3b",
    messages=[
        {
            "role": "user",
            "content": "What is 2 + 2?"
        }
    ]
)

print(response.choices[0].message.content)

2 + 2 equals 4.


In [10]:
!curl http://localhost:11434/v1/models

{"object":"list","data":[{"id":"qwen2.5:3b","object":"model","created":1792267650,"owned_by":"library"}]}


In [11]:
%cd /content/BALROG

import os
os.environ["OPENAI_API_KEY"] = "ollama"

!/content/balrog-venv/bin/python eval.py \
    agent.type=naive \
    agent.max_image_history=0 \
    agent.max_text_history=16 \
    eval.num_workers=1 \
    eval.num_episodes.babyai=3 \
    eval.max_steps_per_episode=20 \
    envs.names=babyai \
    'tasks.babyai_tasks=["BabyAI-MixedTrainLocal-v0/goto"]' \
    client.client_name=vllm \
    client.model_id=qwen2.5:3b \
    client.base_url=http://localhost:11434/v1

/content/BALROG
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/content/BALROG/eval.py:28: Hydra14MigrationWarning: 
version_base="1.1" selects Hydra 1.1 compatibility behavior.
Hydra 1.4 will remove this compatibility behavior.
Before upgrading to Hydra 1.4, set version_base="1.3" and test your application.
See https://hydra.cc/docs/upgrades/1.3_to_1.4/prepare_for_1_4/ for preparation instructions.
  @hydra.main(config_path="balrog/config", config_name="config", version_base="1.1")
/content/balrog-venv/lib/python3.10/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hy

# Testing Groq

In [ ]:
!pip -q install openai requests
!pip -q install vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/88.3 MB 92.6 MB/s eta 0:00:01

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
from openai import OpenAI

groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

# Running openai/gpt-oss-20b model

In [ ]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What is 2 + 2?"}
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

In [ ]:
from pathlib import Path

path = Path("/content/BALROG/balrog/client.py")

code = path.read_text()

old = 'OpenAI(api_key="EMPTY", base_url=self.base_url)'

new = (
    'OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "EMPTY"), '
    'base_url=self.base_url)'
)

if old in code:
    code = code.replace(old, new, 1)
    path.write_text(code)
    print("BALROG authentication updated!")
elif new in code:
    print("Already updated!")
else:
    print("Code differs. Inspect OpenAIWrapper before editing.")

In [ ]:
%cd /content/BALROG

import os
os.environ["OPENAI_API_KEY"] = os.environ["GROQ_API_KEY"]

!/content/balrog-venv/bin/python eval.py \
    agent.type=naive \
    agent.max_image_history=0 \
    agent.max_text_history=16 \
    eval.num_workers=1 \
    eval.num_episodes.babyai=3 \
    eval.max_steps_per_episode=20 \
    envs.names=babyai \
    'tasks.babyai_tasks=["BabyAI-MixedTrainLocal-v0/goto"]' \
    client.client_name=vllm \
    client.model_id=openai/gpt-oss-20b \
    client.base_url=https://api.groq.com/openai/v1

# Testing Openrouter

In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get(
    "OPENROUTER_API_KEY"
)

In [ ]:
import requests

response = requests.get(
    "https://openrouter.ai/api/v1/models"
)

response.raise_for_status()

models = response.json()["data"]

free_models = [
    m for m in models
    if m["id"].endswith(":free")
]

for model in free_models[:20]:
    print(model["id"])

# Running nvidia/nemotron-3.5-super-vl:free model

In [ ]:
from openai import OpenAI

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

response = openrouter_client.chat.completions.create(
    model="nvidia/nemotron-3.5-super-vl:free",
    messages=[
        {"role": "user", "content": "What is 2 + 2?"}
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

Problem encountered: Couldn't use free model because of it is temporarily unavailable due to server overload.

In [ ]:
%cd /content/BALROG

import os
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]

!/content/balrog-venv/bin/python eval.py \
    agent.type=naive \
    agent.max_image_history=0 \
    agent.max_text_history=16 \
    eval.num_workers=1 \
    eval.num_episodes.babyai=3 \
    eval.max_steps_per_episode=20 \
    envs.names=babyai \
    'tasks.babyai_tasks=["BabyAI-MixedTrainLocal-v0/goto"]' \
    client.client_name=vllm \
    client.model_id="nvidia/nemotron-3.5-super-vl:free" \
    client.base_url=https://openrouter.ai/api/v1

# Testing VLLM

In [ ]:
!pip -q install vllm

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import subprocess

vllm_log = open("/content/vllm.log", "w")

vllm_process = subprocess.Popen(
    [
        "vllm", "serve",
        "meta-llama/Llama-3.2-3B-Instruct",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.65"
    ],
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

In [ ]:
import requests
import time

for attempt in range(60):
    try:
        r = requests.get(
            "http://localhost:8000/v1/models",
            timeout=3
        )
        if r.ok:
            print("vLLM server ready!")
            print(r.json())
            break
    except requests.RequestException:
        pass

    if vllm_process.poll() is not None:
        print("vLLM process exited. Check the log.")
        break

    time.sleep(5)
else:
    print("Server not ready. Check /content/vllm.log")

# Running meta-llama/Llama-3.2-3B-Instruct model

In [ ]:
from openai import OpenAI

vllm_client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

response = vllm_client.chat.completions.create(
    model="meta-llama/Llama-3.2-3B-Instruct",
    messages=[
        {"role": "user", "content": "What is 2 + 2?"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
import time
import requests

for i in range(60):
    if vllm_process.poll() is not None:
        print("vLLM exited:", vllm_process.returncode)
        break

    try:
        response = requests.get(
            "http://localhost:8000/v1/models",
            timeout=3
        )

        if response.status_code == 200:
            print("vLLM is ready!")
            print(response.json())
            break

    except requests.RequestException:
        pass

    time.sleep(5)
else:
    print("Server did not become ready.")

if vllm_process.poll() is not None:
    print("\nLast log lines:")
    print(open("/content/vllm.log").read()[-5000:])

# Testing all four models

In [ ]:
clients = {
    "Ollama": (
        ollama_client,
        "qwen2.5:3b"
    ),
    "vLLM": (
        vllm_client,
        "meta-llama/Llama-3.2-3B-Instruct"
    ),
    "Groq": (
        groq_client,
        "openai/gpt-oss-20b"
    ),
    "OpenRouter": (
        openrouter_client,
        OPENROUTER_MODEL
    )
}

for name, (client, model) in clients.items():
    print(f"\nTesting {name}...")

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": "What is 2 + 2?"
                }
            ],
            max_tokens=100
        )

        print("Success!")
        print(response.choices[0].message.content)

    except Exception as e:
        print("Failed:", str(e)[:300])

In [ ]:
import os
import sys
import time
import subprocess
import requests
from pathlib import Path
from google.colab import userdata

# ==========================================================
# 1. CONFIGURATION
# ==========================================================

BALROG_DIR = "/content/BALROG"
PYTHON = "/content/balrog-venv/bin/python"
RESULTS_DIR = Path("/content/balrog_four_runs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Start with 3 episodes to make sure everything works.
# Increase to 100 for the full experiment.
NUM_EPISODES = 3
MAX_STEPS = 20

OPENROUTER_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

clients = {
    "Ollama": {
        "model": "qwen2.5:3b",
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama",
        "local": True
    },
    "vLLM": {
        "model": "meta-llama/Llama-3.2-3B-Instruct",
        "base_url": "http://localhost:8000/v1",
        "api_key": "EMPTY",
        "local": True
    },
    "Groq": {
        "model": "openai/gpt-oss-20b",
        "base_url": "https://api.groq.com/openai/v1",
        "api_key": userdata.get("GROQ_API_KEY"),
        "local": False
    },
    "OpenRouter": {
        "model": OPENROUTER_MODEL,
        "base_url": "https://openrouter.ai/api/v1",
        "api_key": userdata.get("OPENROUTER_API_KEY"),
        "local": False
    }
}

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# ==========================================================
# 2. BALROG AUTHENTICATION FIX
# ==========================================================

# BALROG's vllm client normally uses api_key="EMPTY".
# Allow the currently selected API key to be passed instead.

client_path = Path(BALROG_DIR) / "balrog/client.py"
source = client_path.read_text()

old = 'OpenAI(api_key="EMPTY", base_url=self.base_url)'
new = (
    'OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "EMPTY"), '
    'base_url=self.base_url)'
)

if old in source:
    client_path.write_text(source.replace(old, new, 1))
    print("BALROG authentication patched.")
elif new in source:
    print("BALROG authentication already patched.")
else:
    raise RuntimeError(
        "Could not find the expected client initialization. "
        "Inspect balrog/client.py before continuing."
    )

# ==========================================================
# 3. LOCAL SERVER MANAGEMENT
# ==========================================================

def server_ready(url):
    try:
        r = requests.get(url + "/models", timeout=3)
        return r.status_code == 200
    except requests.RequestException:
        return False


def stop_ollama():
    subprocess.run(
        ["pkill", "-f", "^ollama serve$"],
        check=False
    )
    time.sleep(3)


def start_ollama():
    url = clients["Ollama"]["base_url"]

    if server_ready(url):
        print("Ollama already running.")
        return None

    log = open(RESULTS_DIR / "ollama_server.log", "w")

    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT
    )
    log.close()

    for _ in range(30):
        if server_ready(url):
            print("Ollama ready!")
            return process
        if process.poll() is not None:
            break
        time.sleep(2)

    raise RuntimeError("Ollama failed to start. Check ollama_server.log")


def start_vllm():
    url = clients["vLLM"]["base_url"]

    if server_ready(url):
        print("vLLM already running.")
        return None

    log = open(RESULTS_DIR / "vllm_server.log", "w")

    process = subprocess.Popen(
        [
            "vllm", "serve",
            clients["vLLM"]["model"],
            "--host", "127.0.0.1",
            "--port", "8000",
            "--dtype", "half",
            "--max-model-len", "2048",
            "--gpu-memory-utilization", "0.75",
            "--max-num-seqs", "4"
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        env=os.environ.copy()
    )
    log.close()

    for _ in range(90):
        if server_ready(url):
            print("vLLM ready!")
            return process
        if process.poll() is not None:
            break
        time.sleep(5)

    raise RuntimeError("vLLM failed to start. Check vllm_server.log")


# ==========================================================
# 4. BALROG EVALUATION FUNCTION
# ==========================================================

def evaluate_model(name, config):

    print(f"\n{'=' * 60}")
    print(f"Evaluating: {name}")
    print(f"Model: {config['model']}")
    print(f"{'=' * 60}")

    run_dir = RESULTS_DIR / name.lower()
    run_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env["OPENAI_API_KEY"] = config["api_key"]
    env["HYDRA_FULL_ERROR"] = "1"

    command = [
        PYTHON,
        "eval.py",

        "agent.type=naive",
        "agent.max_image_history=0",
        "agent.max_text_history=16",

        "eval.num_workers=1",
        f"eval.num_episodes.babyai={NUM_EPISODES}",
        f"eval.max_steps_per_episode={MAX_STEPS}",

        "envs.names=babyai",
        'tasks.babyai_tasks=["BabyAI-MixedTrainLocal-v0/goto"]',

        "client.client_name=vllm",
        f"client.model_id={config['model']}",
        f"client.base_url={config['base_url']}",

        f"hydra.run.dir={run_dir}",
    ]

    start = time.time()
    log_path = run_dir / "evaluation.log"

    with open(log_path, "w") as log:
        process = subprocess.run(
            command,
            cwd=BALROG_DIR,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True
        )

    elapsed = time.time() - start

    status = "SUCCESS" if process.returncode == 0 else "FAILED"

    print(f"Status: {status}")
    print(f"Time: {elapsed:.1f} seconds")
    print(f"Log: {log_path}")

    if process.returncode != 0:
        print("\nLast 20 log lines:")
        lines = log_path.read_text(errors="replace").splitlines()
        print("\n".join(lines[-20:]))

    return {
        "API": name,
        "Model": config["model"],
        "Status": status,
        "Runtime_seconds": round(elapsed, 2),
        "Return_code": process.returncode,
        "Log": str(log_path)
    }


# ==========================================================
# 5. RUN ALL FOUR MODELS SEQUENTIALLY
# ==========================================================

results = []

for name, config in clients.items():

    local_process = None

    try:
        # Start the required local inference server
        if name == "Ollama":
            local_process = start_ollama()

        elif name == "vLLM":
            stop_ollama()
            local_process = start_vllm()

        # Run BALROG
        result = evaluate_model(name, config)
        results.append(result)

    except Exception as e:
        print(f"\n{name} FAILED: {e}")

        results.append({
            "API": name,
            "Model": config["model"],
            "Status": "FAILED",
            "Runtime_seconds": None,
            "Return_code": None,
            "Log": str(e)
        })

    finally:
        # Free GPU memory after Ollama and vLLM evaluations
        if local_process is not None:
            local_process.terminate()
            try:
                local_process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                local_process.kill()
                local_process.wait()

        if name == "Ollama":
            stop_ollama()

        print(f"\nFinished {name}")

# ==========================================================
# 6. SAVE SUMMARY
# ==========================================================

import pandas as pd

summary = pd.DataFrame(results)

summary.to_csv(
    RESULTS_DIR / "summary.csv",
    index=False
)

print("\nFINAL SUMMARY")
print(summary.to_string(index=False))
print(f"\nResults saved in: {RESULTS_DIR}")